In [ ]:
# -*- coding: utf-8 -*-
"""
Document (paper/book page) detection with MobileSAM/SAM on CPU, plus perspective rectification.
- NO pip installs in code
- Box-prompt centric MobileSAM/SAM -> geometric scoring -> NMS
- For each detected doc: homography warp (deskew) -> "scanner-like" enhanced outputs
- Saves intermediate images for debugging/inspection
"""

from pathlib import Path
import os, sys, math, random, logging, datetime as dt, json
import numpy as np
import cv2
from PIL import Image, ImageDraw
from IPython.display import display, Image as IPyImage

# =========================
# Global Config (edit here)
# =========================
DATA_DIR     = Path("data/sample3")
MODELS_DIR   = Path("models")
OUTPUT_ROOT  = Path("output")
ALLOWED_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"]

# Models (手動配置)
MOBILE_SAM_CKPT = MODELS_DIR / "mobile_sam.pt"
SAM_CKPTS = [
    MODELS_DIR / "sam_vit_b_01ec64.pth",
    MODELS_DIR / "sam_vit_l_0b3195.pth",
    MODELS_DIR / "sam_vit_h_4b8939.pth",
]

# Inference（CPU・低メモリ）
DEVICE = "cpu"
MAX_SIDE = 1280           # 画像長辺をここに収めて埋め込みを作る
MULTIMASK_PER_BOX = True  # ボックスごとに3候補→最良を採用
BOX_PAD = 8               # ボックスに足すマージン(px, resized座標)

# Box proposals（数を増やすと精度↑／若干時間↑）
GLOBAL_MARGINS = [0.02, 0.05, 0.08, 0.12]
CENTER_SCALES  = [0.90, 0.75, 0.60, 0.50]
CORNER_SCALES  = [0.75, 0.60]
USE_CONTOUR_PROPOSAL = True

# Filters / geometry（厳→緩の2段運用）
AREA_FRAC_MIN_STRICT, AREA_FRAC_MAX_STRICT = 0.04, 0.98
AREA_FRAC_MIN_RELAX , AREA_FRAC_MAX_RELAX  = 0.02, 0.995
RECT_RATIO_MIN_STRICT = 0.80
RECT_RATIO_MIN_RELAX  = 0.72
SOLIDITY_MIN_STRICT    = 0.92
SOLIDITY_MIN_RELAX     = 0.85
AR_MIN, AR_MAX         = 0.30, 4.5
EDGE_SUPPORT_MIN_STRICT = 0.15
EDGE_SUPPORT_MIN_RELAX  = 0.08

NMS_IOU_THR    = 0.55
MAX_FINAL_INST = 4

# Viz
DRAW_POLY_W = 4
DRAW_ALPHA  = 0.35

# CPU固定 & 乱数
# random.seed(7)
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")
os.environ.setdefault("OMP_NUM_THREADS", "4")

# =========================
# Logging
# =========================
logger = logging.getLogger("doc_scan_mobilesam_box_cpu_warp")
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
logger.addHandler(handler)
logger.setLevel(logging.INFO)

# =========================
# Utilities
# =========================
def ensure_dirs():
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    outdir = OUTPUT_ROOT / dt.datetime.now().strftime("%Y%m%d-%H%M%S")
    outdir.mkdir(parents=True, exist_ok=True)
    return outdir

def pick_input_image():
    files = [p for p in DATA_DIR.rglob("*") if p.suffix.lower() in ALLOWED_EXTS]
    if not files:
        raise FileNotFoundError(f"No images found in {DATA_DIR} with {ALLOWED_EXTS}")
    return random.choice(files)

def pil_to_cv(pil):
    return cv2.cvtColor(np.array(pil.convert("RGB")), cv2.COLOR_RGB2BGR)

def cv_to_pil(img):
    return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

def save_and_display(pil_img, outdir, name):
    p = outdir / name
    pil_img.save(p)
    display(IPyImage(filename=str(p)))
    logger.info(f"Saved: {p}")

def draw_polygons(pil_img, polygons, fills=True, boxes=None):
    base = pil_img.convert("RGBA")
    overlay = Image.new("RGBA", base.size, (0,0,0,0))
    draw = ImageDraw.Draw(overlay, "RGBA")
    if boxes:
        for (x1,y1,x2,y2) in boxes:
            col = (255, 200, 0, 140)
            draw.rectangle([x1,y1,x2,y2], outline=col, width=2)
    for poly in polygons:
        if not poly or len(poly) < 3: continue
        col = tuple(np.random.randint(32, 224, size=3).tolist())
        if fills:
            draw.polygon(poly, fill=(*col, int(255*DRAW_ALPHA)))
        draw.line(poly + [poly[0]], width=DRAW_POLY_W, fill=tuple(col)+(255,))
    return Image.alpha_composite(base, overlay).convert("RGB")

# =========================
# Resize & scale
# =========================
def resize_keep_aspect(img_rgb, max_side):
    h, w = img_rgb.shape[:2]
    s = float(max_side) / float(max(h, w))
    if s >= 1.0:
        return img_rgb.copy(), 1.0
    new_w, new_h = int(round(w*s)), int(round(h*s))
    out = cv2.resize(img_rgb, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return out, s

# =========================
# Model loader (MobileSAM優先)
# =========================
def load_mobile_sam_or_sam():
    model = None; api = None; typ = None
    # MobileSAM
    if MOBILE_SAM_CKPT.exists():
        try:
            import torch  # noqa
            from mobile_sam import sam_model_registry as msam_reg
            from mobile_sam import SamPredictor as MSamPred
            model = msam_reg["vit_t"](checkpoint=str(MOBILE_SAM_CKPT))
            model.to(DEVICE); model.eval()
            api = dict(Pred=MSamPred)
            typ = "mobile_sam_vit_t"
            logger.info(f"Loaded MobileSAM: {MOBILE_SAM_CKPT}")
            return model, api, typ
        except Exception as e:
            logger.warning(f"MobileSAM load failed: {e}")

    # SAM (fallback)
    for ck in SAM_CKPTS:
        if ck.exists():
            try:
                import torch  # noqa
                from segment_anything import sam_model_registry as sam_reg
                from segment_anything import SamPredictor as SamPred
                if "vit_b" in ck.name: mtype="vit_b"
                elif "vit_l" in ck.name: mtype="vit_l"
                else: mtype="vit_h"
                model = sam_reg[mtype](checkpoint=str(ck))
                model.to(DEVICE); model.eval()
                api = dict(Pred=SamPred)
                typ = f"sam_{mtype}"
                logger.info(f"Loaded SAM: {ck.name}")
                return model, api, typ
            except Exception as e:
                logger.warning(f"SAM load failed ({ck.name}): {e}")

    raise RuntimeError("No MobileSAM/SAM checkpoint found in models/.")

# =========================
# Box proposals
# =========================
def global_margin_boxes(w, h, margins):
    boxes=[]
    for m in margins:
        x1 = int(round(w*m)); y1=int(round(h*m))
        x2 = int(round(w*(1-m))); y2=int(round(h*(1-m)))
        if x2-x1>10 and y2-y1>10:
            boxes.append([x1,y1,x2,y2])
    return boxes

def center_boxes(w, h, scales):
    cx, cy = w//2, h//2
    boxes=[]
    for s in scales:
        hw, hh = int(round(w*s/2)), int(round(h*s/2))
        x1, y1 = cx-hw, cy-hh
        x2, y2 = cx+hw, cy+hh
        if x2-x1>10 and y2-y1>10:
            boxes.append([x1,y1,x2,y2])
    return boxes

def corner_boxes(w, h, scales):
    boxes=[]
    for s in scales:
        boxes += [
            [0, 0, int(round(w*s)), int(round(h*s))],
            [int(round(w*(1-s))), 0, w-1, int(round(h*s))],
            [0, int(round(h*(1-s))), int(round(w*s)), h-1],
            [int(round(w*(1-s))), int(round(h*(1-s))), w-1, h-1],
        ]
    out=[]
    for (x1,y1,x2,y2) in boxes:
        if x2-x1>10 and y2-y1>10:
            out.append([x1,y1,x2,y2])
    return out

def contour_biggest_box(img_rgb):
    if not USE_CONTOUR_PROPOSAL:
        return []
    g = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    g = cv2.GaussianBlur(g, (3,3), 0)
    edges = cv2.Canny(g, 50, 150)
    edges = cv2.dilate(edges, np.ones((3,3), np.uint8), iterations=1)
    cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return []
    c = max(cnts, key=cv2.contourArea)
    x,y,w,h = cv2.boundingRect(c)
    if w*h < img_rgb.shape[0]*img_rgb.shape[1]*0.03:
        return []
    return [[x,y,x+w,y+h]]

def make_box_proposals(img_rgb, pad=BOX_PAD):
    H,W = img_rgb.shape[:2]
    boxes = []
    boxes += global_margin_boxes(W,H, GLOBAL_MARGINS)
    boxes += center_boxes(W,H, CENTER_SCALES)
    boxes += corner_boxes(W,H, CORNER_SCALES)
    boxes += contour_biggest_box(img_rgb)
    # pad & clip
    padded=[]
    for (x1,y1,x2,y2) in boxes:
        padded.append([
            max(0, x1-pad), max(0, y1-pad),
            min(W-1, x2+pad), min(H-1, y2+pad)
        ])
    # IoU-based dedup
    def iou(a,b):
        ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
        ix1,iy1=max(ax1,bx1),max(ay1,by1)
        ix2,iy2=min(ax2,bx2),min(ay2,by2)
        iw,ih=max(0,ix2-ix1),max(0,iy2-iy1)
        inter=iw*ih; u=(ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
        return inter/max(1,u)
    uniq=[]
    for b in padded:
        keep=True
        for u in uniq:
            if iou(b,u)>0.7:
                keep=False; break
        if keep: uniq.append(b)
    return uniq

def refine_quad_corners(img_bgr, poly, win=15):
    """Refine quad corners by local edge-guided snapping."""
    h, w = img_bgr.shape[:2]
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3,3), 0)
    edges = cv2.Canny(gray, 60, 180)

    refined = []
    for (x, y) in poly:
        x0, x1 = max(0, x - win), min(w, x + win + 1)
        y0, y1 = max(0, y - win), min(h, y + win + 1)
        patch = edges[y0:y1, x0:x1]
        if patch.size == 0 or patch.max() == 0:
            refined.append((x, y))
            continue
        # エッジ密度の最大点を探す（3x3の近傍サーチ）
        best = (x, y); best_cnt = -1
        for yy in range(y0, y1):
            for xx in range(x0, x1):
                if edges[yy, xx] == 0:
                    continue
                y2a, y2b = max(0, yy - 1), min(h, yy + 2)
                x2a, x2b = max(0, xx - 1), min(w, xx + 2)
                cnt = int((edges[y2a:y2b, x2a:x2b] > 0).sum())
                if cnt > best_cnt:
                    best_cnt = cnt
                    best = (xx, yy)
        refined.append(best)

    # 頂点順序を再整列（tl,tr,br,bl）
    pts = np.array(refined, dtype=np.float32)
    s = pts.sum(axis=1); d = np.diff(pts, axis=1)[:,0]
    tl = pts[np.argmin(s)]; br = pts[np.argmax(s)]
    tr = pts[np.argmin(d)]; bl = pts[np.argmax(d)]
    return [(int(tl[0]),int(tl[1])), (int(tr[0]),int(tr[1])),
            (int(br[0]),int(br[1])), (int(bl[0]),int(bl[1]))]


# =========================
# Mask helpers & scoring
# =========================
def polygon_from_mask(mask):
    m = (mask.astype(np.uint8) * 255)
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return None
    c = max(cnts, key=cv2.contourArea)
    rect = cv2.minAreaRect(c)
    box = cv2.boxPoints(rect).astype(int)
    pts = np.array([(int(x), int(y)) for x, y in box], dtype=np.float32)
    s = pts.sum(axis=1); d = np.diff(pts, axis=1)[:,0]
    tl = pts[np.argmin(s)]; br = pts[np.argmax(s)]
    tr = pts[np.argmin(d)]; bl = pts[np.argmax(d)]
    return [(int(tl[0]),int(tl[1])), (int(tr[0]),int(tr[1])),
            (int(br[0]),int(br[1])), (int(bl[0]),int(bl[1]))]

def mask_iou(a, b):
    inter = np.logical_and(a, b).sum()
    uni = np.logical_or(a, b).sum()
    return 0.0 if uni == 0 else float(inter)/float(uni)

def nms_masks(masks, thr=0.55):
    idx = list(range(len(masks)))
    idx.sort(key=lambda i: masks[i].sum(), reverse=True)
    kept=[]
    for i in idx:
        drop=False
        for j in kept:
            if mask_iou(masks[i], masks[j])>thr:
                drop=True; break
        if not drop: kept.append(i)
    return kept

def score_mask(mask, img_bgr, strict=True):
    H,W = img_bgr.shape[:2]
    area_img = float(H*W)
    ys,xs = np.where(mask)
    if xs.size==0: return None
    area = float(xs.size)
    area_frac = area/area_img

    if strict:
        A_MIN, A_MAX = AREA_FRAC_MIN_STRICT, AREA_FRAC_MAX_STRICT
        RECT_MIN = RECT_RATIO_MIN_STRICT
        SOL_MIN  = SOLIDITY_MIN_STRICT
        EDGE_MIN = EDGE_SUPPORT_MIN_STRICT
    else:
        A_MIN, A_MAX = AREA_FRAC_MIN_RELAX, AREA_FRAC_MAX_RELAX
        RECT_MIN = RECT_RATIO_MIN_RELAX
        SOL_MIN  = SOLIDITY_MIN_RELAX
        EDGE_MIN = EDGE_SUPPORT_MIN_RELAX

    if not (A_MIN <= area_frac <= A_MAX):
        return None

    pts = np.stack([xs, ys], axis=1).astype(np.float32)
    rect = cv2.minAreaRect(pts)
    (cx,cy),(rw,rh),_ = rect
    rect_area = max(1.0, float(rw*rh))
    rect_ratio = area/rect_area
    hull = cv2.convexHull(pts)
    hull_area = max(1.0, cv2.contourArea(hull))
    solidity = area/hull_area
    ar = (max(rw, rh) / max(1.0, min(rw, rh))) if min(rw,rh)>0 else 999.0
    if rect_ratio < RECT_MIN or solidity < SOL_MIN or not (AR_MIN <= ar <= AR_MAX):
        return None

    poly = polygon_from_mask(mask)
    edge_support = 0.0
    if poly:
        edges = cv2.Canny(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY), 60, 180)
        total_len=0; hit=0
        for i in range(4):
            x1,y1 = poly[i]; x2,y2 = poly[(i+1)%4]
            seg_len = max(1, int(np.hypot(x2-x1, y2-y1)))
            total_len += seg_len
            for t in np.linspace(0,1,seg_len):
                x = int(round(x1*(1-t)+x2*t)); y=int(round(y1*(1-t)+y2*t))
                if 0<=x<W and 0<=y<H:
                    if (edges[max(0,y-1):min(H,y+2), max(0,x-1):min(W,x+2)]>0).any():
                        hit += 1
        edge_support = hit/max(1,total_len)
    if edge_support < EDGE_MIN:
        return None

    score = 0.5*rect_ratio + 0.25*solidity + 0.25*edge_support
    return dict(score=float(score), poly=poly)

# =========================
# Warp (perspective rectification) & scan-like enhancement
# =========================
def quad_size(poly):
    p = [np.array(pt, dtype=np.float32) for pt in poly]
    w1 = np.linalg.norm(p[1]-p[0]); w2 = np.linalg.norm(p[2]-p[3])
    h1 = np.linalg.norm(p[2]-p[1]); h2 = np.linalg.norm(p[3]-p[0])
    return int(round((w1+w2)/2.0)), int(round((h1+h2)/2.0))

def warp_by_quad(img_bgr, poly, out_scale=1.0, min_side=640, pad=4):
    """Perspective warp with size floor and small padding."""
    # 角を微調整してからワープ
    poly_refined = refine_quad_corners(img_bgr, poly, win=15)
    tl, tr, br, bl = poly_refined

    # 出力サイズ
    def _edge(a, b): return np.linalg.norm(np.array(a, np.float32) - np.array(b, np.float32))
    w_est = 0.5 * (_edge(tl, tr) + _edge(bl, br))
    h_est = 0.5 * (_edge(tr, br) + _edge(tl, bl))
    W = max(int(w_est * out_scale), int(min_side * out_scale))
    H = max(int(h_est * out_scale), int(min_side * out_scale * (h_est / (w_est + 1e-6))))

    # ソース・デスト
    src = np.float32([tl, tr, br, bl])
    dst = np.float32([[pad, pad],
                      [W - 1 - pad, pad],
                      [W - 1 - pad, H - 1 - pad],
                      [pad, H - 1 - pad]])

    M = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(img_bgr, M, (W, H), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return warped


def enhance_scanner_color(img_bgr):
    # 白背景寄せ: LAB-CLAHE + ホワイトバランス + 軽いシャープ
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l,a,b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l2 = clahe.apply(l)
    lab2 = cv2.merge([l2,a,b])
    img2 = cv2.cvtColor(lab2, cv2.COLOR_LAB2BGR)
    # グレーワールド風簡易WB
    avgB,avgG,avgR = [img2[:,:,i].mean() for i in range(3)]
    k = (avgB+avgG+avgR)/3.0 + 1e-6
    gain = [k/avgB, k/avgG, k/avgR]
    wb = cv2.merge([np.clip(img2[:,:,0]*gain[0],0,255),
                    np.clip(img2[:,:,1]*gain[1],0,255),
                    np.clip(img2[:,:,2]*gain[2],0,255)]).astype(np.uint8)
    # 軽いアンシャープ
    blur = cv2.GaussianBlur(wb, (0,0), 1.0)
    sharp = cv2.addWeighted(wb, 1.3, blur, -0.3, 0)
    return sharp

def enhance_scanner_bw(img_bgr):
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.bilateralFilter(g, 7, 40, 40)
    bw = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                               cv2.THRESH_BINARY, 29, 10)
    return cv2.cvtColor(bw, cv2.COLOR_GRAY2BGR)

# =========================
# Main pipeline
# =========================
def run_pipeline(user_image_path=None):
    outdir = ensure_dirs()
    img_path = Path(user_image_path) if user_image_path else pick_input_image()
    logger.info(f"Input: {img_path}")

    pil_orig = Image.open(img_path).convert("RGB")
    img_rgb_orig = np.array(pil_orig)  # H,W,3
    H0,W0 = img_rgb_orig.shape[:2]

    # Resize for predictor
    img_rgb_resized, scale = resize_keep_aspect(img_rgb_orig, MAX_SIDE)
    Hr,Wr = img_rgb_resized.shape[:2]
    inv = 1.0/scale
    logger.info(f"Resize: {W0}x{H0} -> {Wr}x{Hr} (scale={scale:.3f})")

    # Save step: resized
    save_and_display(Image.fromarray(img_rgb_resized), outdir, "step_00_resized.jpg")

    # Load model & predictor
    model, api, model_type = load_mobile_sam_or_sam()
    Pred = api["Pred"]
    predictor = Pred(model)
    predictor.set_image(img_rgb_resized)  # 画像埋め込み（1回のみ）
    logger.info(f"Model: {model_type} (CPU)")

    # ---- Box proposals ----
    boxes = make_box_proposals(img_rgb_resized, pad=BOX_PAD)
    if not boxes:
        boxes = [[BOX_PAD,BOX_PAD,Wr-1-BOX_PAD,Hr-1-BOX_PAD]]
    # Save step: proposals overlay
    step_boxes = draw_polygons(Image.fromarray(img_rgb_resized), [], fills=False, boxes=boxes)
    save_and_display(step_boxes, outdir, "step_01_box_proposals.jpg")

    # ---- SAM predict (box prompt) ----
    img_bgr_resized = cv2.cvtColor(img_rgb_resized, cv2.COLOR_RGB2BGR)
    cand = []
    for bx in boxes:
        box_np = np.array([bx], dtype=np.float32)  # (1,4) xyxy
        masks, scores, _ = predictor.predict(
            box=box_np, multimask_output=MULTIMASK_PER_BOX
        )
        if masks is None: 
            continue
        if MULTIMASK_PER_BOX:
            for k in range(masks.shape[0]):
                m = masks[k].astype(bool)
                s = score_mask(m, img_bgr_resized, strict=True)
                if s is not None and s["poly"] is not None:
                    cand.append((s["score"], m, s))
        else:
            m = masks.astype(bool)
            s = score_mask(m, img_bgr_resized, strict=True)
            if s is not None and s["poly"] is not None:
                cand.append((s["score"], m, s))

    # Relax if needed
    if not cand:
        logger.info("No candidates under strict thresholds; relaxing...")
        for bx in boxes:
            box_np = np.array([bx], dtype=np.float32)
            masks, scores, _ = predictor.predict(
                box=box_np, multimask_output=True
            )
            if masks is None: 
                continue
            for k in range(masks.shape[0]):
                m = masks[k].astype(bool)
                s = score_mask(m, img_bgr_resized, strict=False)
                if s is not None and s["poly"] is not None:
                    cand.append((s["score"], m, s))

    # Force full-frame if still none
    if not cand:
        logger.info("Still no candidates; forcing full-frame box.")
        bx = [BOX_PAD, BOX_PAD, Wr-1-BOX_PAD, Hr-1-BOX_PAD]
        box_np = np.array([bx], dtype=np.float32)
        masks, scores, _ = predictor.predict(box=box_np, multimask_output=True)
        if masks is not None:
            best = None
            for k in range(masks.shape[0]):
                m = masks[k].astype(bool)
                s = score_mask(m, img_bgr_resized, strict=False)
                if s is None: 
                    continue
                if best is None or s["score"] > best[0]:
                    best = (s["score"], m, s)
            if best is not None:
                cand.append(best)

    if not cand:
        logger.warning("No document detected.")
        vis = draw_polygons(pil_orig, [], fills=True)
        save_and_display(vis, outdir, "segmented_overlay.jpg")
        return dict(output_dir=str(outdir), num_polygons=0, polygons=[])

    # ---- Final selection & NMS ----
    cand.sort(key=lambda t: t[0], reverse=True)
    masks_fin = [m for _, m, _ in cand]
    keep = nms_masks(masks_fin, thr=NMS_IOU_THR)[:MAX_FINAL_INST]
    chosen = [cand[i] for i in keep]
    polygons_r = [t[2]["poly"] for t in chosen]

    # Save step: polygons on resized
    step_polys_resized = draw_polygons(Image.fromarray(img_rgb_resized), polygons_r, fills=True)
    save_and_display(step_polys_resized, outdir, "step_02_polygons_resized.jpg")

    # ---- Map polygons back to original & draw overlay ----
    polygons = []
    for poly_r in polygons_r:
        poly_o = [(int(p[0]*(1.0/scale)), int(p[1]*(1.0/scale))) for p in poly_r]
        polygons.append(poly_o)
    overlay = draw_polygons(pil_orig, polygons, fills=True)
    save_and_display(overlay, outdir, "segmented_overlay.jpg")

    # ---- Warp & scanner-like enhancement per instance ----
    img_bgr_orig = pil_to_cv(pil_orig)
    outputs = []
    for i, poly in enumerate(polygons):
        # warp
        warped = warp_by_quad(img_bgr_orig, poly, out_scale=1.0)
        Image.fromarray(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)).save(outdir / f"warp_raw_{i}.png")
        # enhance color
        color = enhance_scanner_color(warped)
        Image.fromarray(cv2.cvtColor(color, cv2.COLOR_BGR2RGB)).save(outdir / f"warp_color_{i}.png")
        # enhance bw
        bw = enhance_scanner_bw(warped)
        Image.fromarray(cv2.cvtColor(bw, cv2.COLOR_BGR2RGB)).save(outdir / f"warp_bw_{i}.png")
        outputs.append(dict(index=i,
                            polygon=poly,
                            warp_raw=str((outdir / f"warp_raw_{i}.png").name),
                            warp_color=str((outdir / f"warp_color_{i}.png").name),
                            warp_bw=str((outdir / f"warp_bw_{i}.png").name)))

    # 表示（上位1枚のサマリ）
    if outputs:
        display(IPyImage(filename=str(outdir / outputs[0]["warp_color"])))
        display(IPyImage(filename=str(outdir / outputs[0]["warp_bw"])))

    # 追加のステップ画像（任意）
    # 画像全体＋候補ボックス、最終多角形はすでに保存済み

    result = dict(output_dir=str(outdir),
                  num_polygons=len(polygons),
                  polygons=polygons,
                  warped=outputs)
    logger.info(json.dumps(result, ensure_ascii=False, indent=2))
    return result

# =========================
# Run (for notebooks)
# =========================
if __name__ == "__main__":
    info = run_pipeline(user_image_path=None)
    print(json.dumps(info, ensure_ascii=False, indent=2))
